# Ajout de la commune, département et région française

L'objectif de ce notebook est d'ajouter les informations géographiques à nos mesures météorologiques et de radiations.

Cela se fera à travers des jointures spatiales sur les données GeoJSON disponible sur le [site](https://www.data.gouv.fr/datasets/contours-administratifs) data.gouv.fr du gouvernement.

In [10]:
import pyspark.sql.functions as F

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 85, Finished, Available, Finished, False)

## Lecture des données

In [1]:
GEOJSON_PATH = "Files/FranceGeo/"

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 5, Finished, Available, Finished, False)

In [2]:
# Chargement des mesures au sol à ajouter ces informations géographiques
df_measures = spark.sql("SELECT * FROM openradiation_lakehouse.silver.ground_measures")

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 6, Finished, Available, Finished, False)

In [3]:
#display(df_measures)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a547d06-e76c-422d-962d-ba273058d832)

In [4]:
# Chargement des mesures météorologiques
df_meteo = spark.sql("SELECT * FROM Lakehouse_meteo_France.dbo.silver_meteo_france")

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 8, Finished, Available, Finished, False)

In [5]:
#display(df_meteo)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5aee6385-ba0a-4d5f-bc63-cbf1a289bd85)

Pour pouvoir lire les points spatio-temporels de nos fichiers GeoJSON, nous allons utiliser l'extension Sedona qui facilite la tâche dans ce processus.

In [6]:
import sys

from sedona.spark import SedonaContext

# Informations sur l'environnement
#print("Spark :", spark.version)
#print("Scala :", spark._jvm.scala.util.Properties.versionNumberString())
#print("Java :", spark._jvm.java.lang.System.getProperty("java.version"))
#print("Python :", sys.version)

# Initialisation Sedona
sedona = SedonaContext.create(spark)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 10, Finished, Available, Finished, False)

In [7]:
# Lecture des fichiers GeoJSON
df_communes = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"communes-1000m.geojson")
)

df_departements = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"departements-1000m.geojson")
)

df_regions = (
    sedona.read
    .format("geojson")
    .load(GEOJSON_PATH+"regions-1000m.geojson")
)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 11, Finished, Available, Finished, False)

In [8]:
#print("Extrait communes : ")
#df_communes.show()

#print("Extrait départements :")
#df_departements.show()

#print("Extrait régions :")
#df_regions.show()

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 82, Finished, Available, Finished, False)

Extrait communes : 
+--------------------+-----------------+
|            features|             type|
+--------------------+-----------------+
|[{POLYGON ((4.958...|FeatureCollection|
+--------------------+-----------------+

Extrait départements :
+--------------------+-----------------+
|            features|             type|
+--------------------+-----------------+
|[{POLYGON ((5.825...|FeatureCollection|
+--------------------+-----------------+

Extrait régions :
+--------------------+-----------------+
|            features|             type|
+--------------------+-----------------+
|[{MULTIPOLYGON ((...|FeatureCollection|
+--------------------+-----------------+



Les GeoJSON ont bien été lues, plus qu'à les préparer

## Préparation des données GeoJSON

In [11]:
# Extrayons de notre liste nos lignes de données
df_communes_exploded = (
    df_communes
    .select(F.explode("features").alias("feature"))
)

# Sélectionnons les colonnes qu'on veut de nos features
df_communes_geo = df_communes_exploded.select(
    F.col("feature.geometry").alias("geom_commune"),
    F.col("feature.properties.code").alias("code_commune"),
    F.col("feature.properties.nom").alias("nom_commune"),
    F.col("feature.properties.region").alias("code_region"),
    F.col("feature.properties.departement").alias("code_departement"),
).withColumn("geom_commune", F.expr("ST_MakeValid(geom_commune)"))

# Faisons la même chose pour le département et la région
df_departements_geo = (
    df_departements
    .select(F.explode("features").alias("feature"))
    .select(
        F.col("feature.geometry").alias("geom_departement"),
        F.col("feature.properties.code").alias("code_departement"),
        F.col("feature.properties.nom").alias("nom_departement")
    ).withColumn("geom_departement", F.expr("ST_MakeValid(geom_departement)"))
)

df_regions_geo = (
    df_regions
    .select(F.explode("features").alias("feature"))
    .select(
        F.col("feature.geometry").alias("geom_region"),
        F.col("feature.properties.code").alias("code_region"),
        F.col("feature.properties.nom").alias("nom_region")
    ).withColumn("geom_region", F.expr("ST_MakeValid(geom_region)"))
)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 86, Finished, Available, Finished, False)

In [33]:
# Jointure des communes avec les départements
df_communes_departements = (
    df_communes_geo
    .join(
        df_departements_geo,
        on="code_departement",
        how="left"
    )
)

# Jointure avec les régions
df_communes_complet = (
    df_communes_departements
    .join(
        df_regions_geo,
        on="code_region",
        how="left"
    )
)

# Mise en cache du DataFrame final
df_communes_complet = df_communes_complet.cache()

# Déclenchement du calcul pour matérialiser le cache
print(df_communes_complet.count())

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 114, Finished, Available, Finished, False)

35014


In [34]:
#display(df_communes_complet)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 115, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d2cf1bb-fe35-4047-ab0a-a077d08d4113)

Données prêtes à être utilisées, préparons nos données concernant les mesures.

## Préparation des données de mesures

### Pour les mesures de radiations

In [15]:
df_measures_prep = df_measures.drop("qualification")

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 90, Finished, Available, Finished, False)

### Pour les mesures météorologiques

In [18]:
df_meteo_prep = df_meteo.drop("source", "data")

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 93, Finished, Available, Finished, False)

Maintenant que les mesures sont prêtes, faisons la jointure spatiale.

## Jointure spatiale pour obtenir la région, le département et la commune selon les coordonnées

D'abord, nous allons créer la colonne `geom_point` permettant de faire la jointure spatiale.

Ensuite, nous allons faire les jointures avec les communes, départements et régions

Enfin, une dernière jointure sera faite pour relier les mesures avec ses informations géographiques grâce à leurs coordonnées.

In [40]:
def jointure_comm_dep_reg(df):
    """
    Permets d'ajouter les informations géographiques avec la latitude et la longitude.

    Args:
        df: pyspark.DataFrame : Dataframe contenant les colonnes "latitude" et "longitude"
    
    Returns:
        df_finale: pyspark.DataFrame : DataFrame avec l'ajout des informations géographiques
    """
    # Capture des coordonnées uniques
    df_measures_point = (
    df
    .select("latitude", "longitude")
    .dropDuplicates()
    .withColumn(
        "geom_point",
        F.expr("""
            ST_Point(
                CAST(longitude AS DOUBLE),
                CAST(latitude AS DOUBLE)
            )
        """)
    )
    )

    # Jointure pour les communes
    df_france_communes = (
    df_measures_point.alias("f")
    .join(
        df_communes_complet.alias("c"),
        F.expr("ST_Intersects(f.geom_point, c.geom_commune)"),
        "left"
    )
    .select(
        "f.*",
        F.col("c.code_commune"),
        F.col("c.nom_commune"),
        F.col("c.code_region"),
        F.col("c.nom_region"),
        F.col("c.code_departement"),
        F.col("c.nom_departement")
    )
    )

   
    # Jointure finale pour relier les informations géographiques avec les mesures
    df_finale = (
    df
    .alias("f")
    .join(
        df_france_communes.alias("fr"),
        ["latitude", "longitude"],
        "left"
    )
    .select(
        "f.*",
        F.col("fr.code_commune"),
        F.col("fr.nom_commune"),
        F.col("fr.code_departement"),
        F.col("fr.nom_departement"),
        F.col("fr.code_region"),
        F.col("fr.nom_region")
    )
    .filter("nom_region IS NOT NULL")
    )

    return df_finale

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 121, Finished, Available, Finished, False)

In [41]:
df_measures_fr = jointure_comm_dep_reg(df_measures_prep)
df_meteo_fr = jointure_comm_dep_reg(df_meteo_prep)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 122, Finished, Available, Finished, False)

In [42]:
# Vérification du succès de l'opération
#display(df_measures_fr)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 123, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec129e8a-02c9-45d2-a919-a7bc2671a61e)

In [43]:
# Vérification du succès de l'opération
#display(df_meteo_fr)

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 124, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c49d675-6d6f-49b8-95a8-0ccaf4e818d3)

Maintenant que les mesures ont leurs positions géographiques.

Ecrivons leurs résultats dans deux tables silver dans leurs lakehouses respectives.

In [44]:
# Ecriture des résultats
df_measures_fr.write.format("delta").mode("overwrite").saveAsTable("openradiation_lakehouse.silver.french_ground_measures")
df_meteo_fr.write.format("delta").mode("overwrite").saveAsTable("Lakehouse_meteo_France.dbo.silver_french_meteo_measure")

StatementMeta(, efda3fed-cdff-4988-b1ef-0aed8b55af05, 127, Finished, Available, Finished, False)